In [70]:
# Recherche et sélection NASA PSI

from pathlib import Path
import os
import json
import hashlib
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone


def initialize_project():

    project = Path("/content/FLARE_AI")

    directories = {
        "raw": project / "data/raw",
        "microfire": project / "data/raw/microfire_atlas",
        "nasa": project / "data/raw/nasa_originals",
        "processed": project / "data/processed",
        "experiments": project / "data/processed/experiments",
        "metadata": project / "data/processed/metadata",
        "publications": project / "data/processed/publications",
        "knowledge": project / "knowledge_base",
        "agents": project / "agents",
        "tests": project / "tests",
        "manifests": project / "manifests",
        "reports": project / "reports"
    }

    for directory in directories.values():
        directory.mkdir(parents=True, exist_ok=True)

    print("FLARE AI project initialized successfully.")

    return project, directories


# Execute initialization
PROJECT, DIRS = initialize_project()

FLARE AI project initialized successfully.


In [71]:
# Catalogue NASA PSI

def build_investigation_catalog(directories):

    investigations = [
        {
            "name": "BASS-II",
            "psi_id": "PSI-25",
            "topic": "Solid fuel combustion",
            "priority": 1,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-25"
        },
        {
            "name": "FLEX",
            "psi_id": "PSI-69",
            "topic": "Droplet combustion",
            "priority": 1,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-69"
        },
        {
            "name": "SAFFIRE-I",
            "psi_id": "PSI-98",
            "topic": "Spacecraft fire safety",
            "priority": 1,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-98"
        },
        {
            "name": "SAFFIRE-II",
            "psi_id": "PSI-99",
            "topic": "Spacecraft fire safety",
            "priority": 1,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-99"
        },
        {
            "name": "SAFFIRE-III",
            "psi_id": "PSI-100",
            "topic": "Spacecraft fire safety",
            "priority": 1,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-100"
        },
        {
            "name": "SAME-R",
            "psi_id": "PSI-101",
            "topic": "Smoke and aerosols",
            "priority": 2,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-101"
        },
        {
            "name": "SAME",
            "psi_id": "PSI-102",
            "topic": "Smoke and aerosols",
            "priority": 2,
            "url": "https://psi.nasa.gov/physci/repo/data/investigations/PSI-102"
        }
    ]

    catalog = {
        "project": "FLARE AI",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "investigations": investigations
    }

    output = directories["knowledge"] / "investigation_catalog.json"

    output.write_text(
        json.dumps(catalog, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"{len(investigations)} investigations catalogued.")

    return investigations


INVESTIGATIONS = build_investigation_catalog(DIRS)




7 investigations catalogued.


In [72]:
# Vérification des sources NASA
def verify_nasa_sources(investigations, timeout=20):
    results = []

    for investigation in investigations:
        url = investigation["url"]

        result = {
            "name": investigation["name"],
            "psi_id": investigation["psi_id"],
            "url": url,
            "status_code": None,
            "accessible": False,
            "error": None
        }

        try:
            response = requests.get(
                url,
                timeout=timeout,
                headers={"User-Agent": "FLARE-AI-Research/1.0"}
            )

            result["status_code"] = response.status_code
            result["accessible"] = response.status_code == 200

        except requests.RequestException as error:
            result["error"] = str(error)

        results.append(result)

    output = DIRS["metadata"] / "nasa_source_verification.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=4, ensure_ascii=False)

    print(pd.DataFrame(results).to_string(index=False))
    return results


NASA_SOURCE_RESULTS = verify_nasa_sources(INVESTIGATIONS)

       name  psi_id                                                          url  status_code  accessible error
    BASS-II  PSI-25  https://psi.nasa.gov/physci/repo/data/investigations/PSI-25          200        True  None
       FLEX  PSI-69  https://psi.nasa.gov/physci/repo/data/investigations/PSI-69          200        True  None
  SAFFIRE-I  PSI-98  https://psi.nasa.gov/physci/repo/data/investigations/PSI-98          200        True  None
 SAFFIRE-II  PSI-99  https://psi.nasa.gov/physci/repo/data/investigations/PSI-99          200        True  None
SAFFIRE-III PSI-100 https://psi.nasa.gov/physci/repo/data/investigations/PSI-100          200        True  None
     SAME-R PSI-101 https://psi.nasa.gov/physci/repo/data/investigations/PSI-101          200        True  None
       SAME PSI-102 https://psi.nasa.gov/physci/repo/data/investigations/PSI-102          200        True  None


In [73]:
# Dataset prepared
DATASETS = {
    "bass2_tests.csv":
        "https://microfire-atlas.vercel.app/downloads/bass2-tests.csv",

    "saffire_runs.csv":
        "https://microfire-atlas.vercel.app/downloads/saffire-runs.csv",

    "luci_runs.csv":
        "https://microfire-atlas.vercel.app/downloads/luci-runs.csv",

    "findings.json":
        "https://microfire-atlas.vercel.app/downloads/findings.json",

    "source_manifest.json":
        "https://microfire-atlas.vercel.app/downloads/source-manifest.json",

    "flame_vision.csv":
        "https://microfire-atlas.vercel.app/downloads/flame-vision.csv",

    "evidence_graph.json":
        "https://microfire-atlas.vercel.app/downloads/evidence-graph.json"
}


def download_microfire_datasets(datasets, directories):
    results = []

    for filename, url in datasets.items():
        destination = directories["microfire"] / filename

        result = {
            "filename": filename,
            "url": url,
            "downloaded": False,
            "size_bytes": 0,
            "sha256": None,
            "error": None
        }

        try:
            response = requests.get(
                url,
                timeout=60,
                headers={"User-Agent": "FLARE-AI-Research/1.0"}
            )
            response.raise_for_status()

            content = response.content
            content_type = response.headers.get("Content-Type", "").lower()

            if not content:
                raise ValueError("Empty response.")

            if "text/html" in content_type or content.lstrip().lower().startswith(
                (b"<!doctype html", b"<html")
            ):
                raise ValueError("The URL returned an HTML page, not a dataset.")

            if filename.endswith(".json"):
                json.loads(content.decode("utf-8-sig"))

            if filename.endswith(".csv"):
                pd.read_csv(destination if destination.exists() else
                            __import__("io").BytesIO(content), nrows=5)

            destination.write_bytes(content)

            result["downloaded"] = True
            result["size_bytes"] = len(content)
            result["sha256"] = hashlib.sha256(content).hexdigest()

        except Exception as error:
            result["error"] = str(error)

        results.append(result)

    output = directories["manifests"] / "microfire_download_manifest.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=4, ensure_ascii=False)

    print(pd.DataFrame(results).to_string(index=False))
    return results


DOWNLOAD_RESULTS = download_microfire_datasets(DATASETS, DIRS)

            filename                                                               url  downloaded  size_bytes                                                           sha256                                                                                              error
     bass2_tests.csv      https://microfire-atlas.vercel.app/downloads/bass2-tests.csv        True       10971 dc148c0b8c85a0eabf4b99f0d9db6bc9ca46a82b22e5d94398a84dd9399b04a8                                                                                               None
    saffire_runs.csv     https://microfire-atlas.vercel.app/downloads/saffire-runs.csv        True        3717 da5a722b281d5053730d68175407f999182fef7aa9d39843e81c23924ff04014                                                                                               None
       luci_runs.csv        https://microfire-atlas.vercel.app/downloads/luci-runs.csv        True         959 041a2211cde07fdd3f978f03530c6db6975a8bb34becd0dbe53ef184ceeb64fc

In [74]:
# Importation

import zipfile
import shutil

def import_nasa_files(directories):
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("This function must be run in Google Colab.")

    uploaded_files = files.upload()
    imported = []

    for filename, content in uploaded_files.items():
        destination = directories["nasa"] / filename
        destination.write_bytes(content)

        record = {
            "filename": filename,
            "path": str(destination),
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "extracted": False
        }

        if filename.lower().endswith(".zip"):
            extract_dir = directories["nasa"] / Path(filename).stem
            extract_dir.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(destination, "r") as archive:
                archive.extractall(extract_dir)

            record["extracted"] = True
            record["extract_directory"] = str(extract_dir)

        imported.append(record)

    output = directories["manifests"] / "nasa_import_manifest.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(imported, file, indent=4, ensure_ascii=False)

    print(f"{len(imported)} files imported.")
    return imported


# Run when you have downloaded NASA files from PSI:
# NASA_IMPORTED_FILES = import_nasa_files(DIRS)

In [80]:
# Recherche des publications
from datetime import datetime, timezone
def search_nasa_publications(
    queries=None,
    page_size=50,
    max_pages=2
):
    if queries is None:
        queries = [
            "microgravity combustion",
            "spacecraft fire safety",
            "solid fuel combustion microgravity",
            "SAFFIRE spacecraft fire",
            "droplet combustion microgravity",
            "smoke aerosol microgravity"
        ]

    endpoint = "https://ntrs.nasa.gov/api/citations/search"
    publications = []

    for query in queries:
        for page in range(max_pages):
            params = {
                "q": query,
                "page": page,
                "pageSize": page_size
            }

            try:
                response = requests.get(
                    endpoint,
                    params=params,
                    timeout=30,
                    headers={"User-Agent": "FLARE-AI-Research/1.0"}
                )
                response.raise_for_status()
                payload = response.json()

                if isinstance(payload, list):
                    items = payload
                else:
                    items = (
                        payload.get("results")
                        or payload.get("items")
                        or payload.get("documents")
                        or []
                    )

                if not items:
                    break

                for item in items:
                    publications.append({
                        "query": query,
                        "ntrs_id": item.get("id") or item.get("ntrsId"),
                        "title": item.get("title"),
                        "abstract": item.get("abstract"),
                        "authors": item.get("authors"),
                        "publication_date": (
                            item.get("publicationDate")
                            or item.get("issueDate")
                        ),
                        "url": (
                            "https://ntrs.nasa.gov/citations/"
                            + str(item.get("id") or item.get("ntrsId"))
                        )
                    })

            except Exception as error:
                print(f"Search error for '{query}': {error}")
                break

            time.sleep(0.5)

    unique = {}
    for publication in publications:
        key = publication.get("ntrs_id") or publication.get("title")
        if key:
            unique[key] = publication

    results = list(unique.values())

    output = DIRS["publications"] / "nasa_publications.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=4, ensure_ascii=False)

    print(f"{len(results)} unique publication records collected.")
    return results


NASA_PUBLICATIONS = search_nasa_publications()

47 unique publication records collected.


In [62]:
# Inventaire de tous les datasets
def build_dataset_inventory(directories):
    inventory = []

    allowed_extensions = {
        ".csv", ".json", ".xlsx", ".xls",
        ".pdf", ".txt", ".zip", ".xml"
    }

    for root_name in ["microfire", "nasa"]:
        root = directories[root_name]

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in allowed_extensions:
                continue

            inventory.append({
                "filename": path.name,
                "extension": path.suffix.lower(),
                "source_folder": root_name,
                "relative_path": str(path.relative_to(directories["raw"])),
                "size_bytes": path.stat().st_size,
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
            })

    output = directories["metadata"] / "dataset_inventory.csv"

    pd.DataFrame(inventory).to_csv(output, index=False)

    print(f"{len(inventory)} files inventoried.")
    return pd.DataFrame(inventory)


DATASET_INVENTORY = build_dataset_inventory(DIRS)

6 files inventoried.


In [63]:
# Analyse de la qualité des données

def generate_quality_report(directories):

    reports = []

    for path in directories["raw"].rglob("*.csv"):

        try:
            df = pd.read_csv(path)

            report = {
                "filename": path.name,
                "rows": len(df),
                "columns": len(df.columns),
                "duplicate_rows": int(df.duplicated().sum()),
                "missing_cells": int(df.isna().sum().sum()),
                "missing_percentage": round(
                    df.isna().sum().sum() / max(df.size, 1) * 100, 2
                ),
                "columns_details": {}
            }

            for column in df.columns:

                report["columns_details"][str(column)] = {
                    "dtype": str(df[column].dtype),
                    "missing": int(df[column].isna().sum()),
                    "unique_values": int(
                        df[column].nunique(dropna=True)
                    )
                }

            reports.append(report)

        except Exception as error:

            reports.append({
                "filename": path.name,
                "error": str(error)
            })

    output = directories["reports"] / "data_quality_report.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(reports, file, indent=4, ensure_ascii=False)

    print(f"Quality report generated for {len(reports)} CSV files.")

    return reports


# Execute function
QUALITY_REPORT = generate_quality_report(DIRS)

Quality report generated for 3 CSV files.


In [64]:
# Cleaning and normalization of datasets
import re
def normalize_column_name(column):

    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)

    return column.strip("_")


def clean_and_normalize_datasets(directories):

    processed_files = []

    csv_files = list(directories["raw"].rglob("*.csv"))

    print(f"Found {len(csv_files)} raw CSV files.")

    for path in csv_files:

        try:
            df = pd.read_csv(path)

            original_rows = len(df)
            original_columns = list(df.columns)

            df.columns = [
                normalize_column_name(column)
                for column in df.columns
            ]

            cleaned_df = df.drop_duplicates().copy()

            destination = (
                directories["experiments"]
                / f"{path.stem}_cleaned.csv"
            )

            cleaned_df.to_csv(destination, index=False)

            processed_files.append({
                "source": str(path),
                "output": str(destination),
                "original_rows": original_rows,
                "processed_rows": len(cleaned_df),
                "original_columns": original_columns,
                "normalized_columns": list(cleaned_df.columns),
                "removed_duplicates": original_rows - len(cleaned_df)
            })

            print(
                f"SUCCESS: {path.name} | "
                f"{len(cleaned_df)} rows | "
                f"{len(cleaned_df.columns)} columns"
            )

        except Exception as error:

            print(f"ERROR: {path.name}: {error}")

    output = directories["metadata"] / "cleaning_manifest.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(
            processed_files,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(f"\nProcessed datasets: {len(processed_files)}")

    return processed_files


CLEANING_RESULTS = clean_and_normalize_datasets(DIRS)

Found 3 raw CSV files.
SUCCESS: luci_runs.csv | 2 rows | 19 columns
SUCCESS: bass2_tests.csv | 56 rows | 17 columns
SUCCESS: saffire_runs.csv | 20 rows | 23 columns

Processed datasets: 3


In [75]:
# Génération du Data Dictionary
def build_data_dictionary(directories):
    dictionary = []

    for path in directories["experiments"].glob("*.csv"):
        try:
            df = pd.read_csv(path)

            for column in df.columns:
                series = df[column]

                dictionary.append({
                    "dataset": path.name,
                    "column": column,
                    "data_type": str(series.dtype),
                    "description": None,
                    "unit": None,
                    "missing_values": int(series.isna().sum()),
                    "unique_values": int(series.nunique(dropna=True)),
                    "sample_values": [
                        str(value)
                        for value in series.dropna().head(5).tolist()
                    ]
                })

        except Exception as error:
            print(f"Could not inspect {path.name}: {error}")

    output = directories["metadata"] / "data_dictionary.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(dictionary, file, indent=4, ensure_ascii=False)

    print(f"{len(dictionary)} column definitions generated.")
    return dictionary


DATA_DICTIONARY = build_data_dictionary(DIRS)

59 column definitions generated.


In [76]:
# Registre des sources et provenance
def build_source_registry(
    investigations,
    publications,
    download_results,
    imported_files
):
    registry = []

    for investigation in investigations:
        registry.append({
            "source_name": investigation["name"],
            "source_type": "NASA PSI investigation",
            "identifier": investigation["psi_id"],
            "url": investigation["url"],
            "provenance": "NASA",
            "verification_status": "Needs individual data verification"
        })

    for publication in publications:
        registry.append({
            "source_name": publication.get("title"),
            "source_type": "NASA NTRS publication",
            "identifier": publication.get("ntrs_id"),
            "url": publication.get("url"),
            "provenance": "NASA NTRS",
            "verification_status": "Bibliographic record collected"
        })

    for dataset in download_results:
        registry.append({
            "source_name": dataset["filename"],
            "source_type": "Dataset",
            "identifier": dataset["sha256"],
            "url": dataset["url"],
            "provenance": "Third-party MicroFire Atlas",
            "verification_status": (
                "Downloaded; NASA cross-check required"
                if dataset["downloaded"]
                else "Download failed"
            )
        })

    for item in imported_files:
        registry.append({
            "source_name": item["filename"],
            "source_type": "NASA original file",
            "identifier": item["sha256"],
            "url": None,
            "provenance": "User-imported NASA file",
            "verification_status": "Imported; citation review required"
        })

    output = DIRS["metadata"] / "source_registry.json"

    with open(output, "w", encoding="utf-8") as file:
        json.dump(registry, file, indent=4, ensure_ascii=False)

    print(f"{len(registry)} source records registered.")
    return registry


SOURCE_REGISTRY = build_source_registry(
    INVESTIGATIONS,
    NASA_PUBLICATIONS,
    DOWNLOAD_RESULTS,
    []  # Replace with NASA_IMPORTED_FILES after importing NASA files.
)

61 source records registered.


In [96]:

class ResearchAgent:

    def __init__(self, investigations, publications):
        self.investigations = investigations
        self.publications = publications

    def search_investigations(self, query):
        query = query.lower()
        results = []

        for investigation in self.investigations:
            searchable_text = " ".join(
                str(value)
                for value in investigation.values()
            ).lower()

            if query in searchable_text:
                results.append(investigation)

        return results

    def search_publications(self, query):
        query = query.lower()
        results = []

        for publication in self.publications:
            searchable = " ".join([
                str(publication.get("title", "")),
                str(publication.get("abstract", "")),
                str(publication.get("authors", ""))
            ]).lower()

            if query in searchable:
                results.append(publication)

        return results

    def research(self, query):
        return {
            "query": query,
            "investigations": self.search_investigations(query),
            "publications": self.search_publications(query),
            "generated_at": datetime.now(timezone.utc).isoformat()
        }


def initialize_research_agent(investigations, publications):
    agent = ResearchAgent(investigations, publications)
    print("Research Agent initialized.")
    return agent


RESEARCH_AGENT = initialize_research_agent(
    INVESTIGATIONS,
    NASA_PUBLICATIONS
)


Research Agent initialized.


In [78]:
# Data Agent
class DataAgent:

    def __init__(self, directories):
        self.directories = directories

    def list_datasets(self):
        datasets = []

        for path in self.directories["raw"].rglob("*"):
            if path.is_file():
                datasets.append({
                    "filename": path.name,
                    "path": str(path),
                    "extension": path.suffix.lower(),
                    "size_bytes": path.stat().st_size
                })

        return datasets

    def load_csv(self, filename):
        possible_paths = [
            self.directories["microfire"] / filename,
            self.directories["nasa"] / filename,
            self.directories["experiments"] / filename
        ]

        for path in possible_paths:
            if path.exists():
                return pd.read_csv(path)

        raise FileNotFoundError(f"Dataset not found: {filename}")

    def profile_dataset(self, filename):
        df = self.load_csv(filename)

        return {
            "filename": filename,
            "rows": len(df),
            "columns": list(df.columns),
            "missing_values": df.isna().sum().to_dict(),
            "duplicate_rows": int(df.duplicated().sum()),
            "statistics": df.describe(include="all").replace(
                {np.nan: None}
            ).to_dict()
        }

    def search_files(self, query):
        query = query.lower()

        return [
            item for item in self.list_datasets()
            if query in item["filename"].lower()
        ]

    def compare_datasets(self, filename_a, filename_b):
        df_a = self.load_csv(filename_a)
        df_b = self.load_csv(filename_b)

        return {
            "dataset_a": filename_a,
            "dataset_b": filename_b,
            "columns_a": list(df_a.columns),
            "columns_b": list(df_b.columns),
            "shared_columns": sorted(
                set(df_a.columns).intersection(df_b.columns)
            ),
            "rows_a": len(df_a),
            "rows_b": len(df_b)
        }


def initialize_data_agent(directories):
    agent = DataAgent(directories)
    print("Data Agent initialized.")
    return agent


DATA_AGENT = initialize_data_agent(DIRS)

# Examples:
# DATA_AGENT.list_datasets()
# DATA_AGENT.profile_dataset("bass2_tests.csv")
# DATA_AGENT.search_files("saffire")

Data Agent initialized.


In [81]:
def inspect_microfire_datasets(directories):

    for path in directories["microfire"].glob("*.csv"):

        print("\n" + "=" * 60)
        print(f"DATASET: {path.name}")
        print("=" * 60)

        try:
            df = pd.read_csv(path)

            print("Shape:", df.shape)
            print("\nColumns:")
            print(df.columns.tolist())

            print("\nFirst 3 rows:")
            display(df.head(3))

        except Exception as error:
            print("Error:", error)


inspect_microfire_datasets(DIRS)


DATASET: luci_runs.csv
Shape: (2, 19)

Columns:
['id', 'sample', 'material', 'material_verbatim', 'size_verbatim', 'direction', 'gravity', 'gravity_note', 'o2_start_pct', 'o2_end_pct', 'pressure_kpa', 'pressure_basis', 'spread_base_mm_s', 'spread_tip_mm_s', 'outcome', 'outcome_label', 'source_id', 'outcome_pdf_page', 'outcome_quote']

First 3 rows:


,id,sample,material,material_verbatim,size_verbatim,direction,gravity,gravity_note,o2_start_pct,o2_end_pct,pressure_kpa,pressure_basis,spread_base_mm_s,spread_tip_mm_s,outcome,outcome_label,source_id,outcome_pdf_page,outcome_quote
0,luci-sibal,SIBAL,SIBAL fabric,cotton-fiberglass blend,5 cm wide x 30 cm,downward,lunar,Simulated lunar gravity on a spinning sounding...,20,18.0,101.3,stated as normal pressure,0.92,1.1,sustained,Spread steadily for over a minute,luci,16,The flame spread for over a minute.
1,luci-pmma,PMMA rod,PMMA,4-mm diameter rod of acrylic plastic,4-mm diameter rod,upward,lunar,Simulated lunar gravity on a spinning sounding...,20,14.3,101.3,stated as normal pressure,NaN,1.5,extinguished,"Spread about 6 cm, then went out",luci,20,Total base spread of ~6cm until extinction.



DATASET: bass2_tests.csv
Shape: (56, 17)

Columns:
['id', 'test_id', 'investigation', 'material', 'thickness_mm', 'width_mm', 'flow_direction', 'flow_initial_cm_s', 'flow_final_cm_s', 'flow_as_recorded', 'oxygen_vol_pct', 'pressure_kpa', 'outcome', 'outcome_label', 'nasa_notes', 'source_id', 'pdf_page']

First 3 rows:


,id,test_id,investigation,material,thickness_mm,width_mm,flow_direction,flow_initial_cm_s,flow_final_cm_s,flow_as_recorded,oxygen_vol_pct,pressure_kpa,outcome,outcome_label,nasa_notes,source_id,pdf_page
0,sibal-GMT45-T1,GMT45-T1,BASS-II,SIBAL fabric,0.32,22.0,concurrent,10.0,10.0,10,18.5,101.325,burned_outcome_not_stated,Burned; final outcome not stated in table,NaN,bass2-summary,104
1,sibal-GMT45-T2,GMT45-T2,BASS-II,SIBAL fabric,0.32,22.0,concurrent,10.0,5.0,10 to 5,18.5,101.325,burned_outcome_not_stated,Burned; final outcome not stated in table,NaN,bass2-summary,104
2,sibal-GMT45-T4,GMT45-T4,BASS-II,SIBAL fabric,0.32,22.0,concurrent,10.0,2.2,10 to 2.2,18.7,101.325,quenched_low_flow,Quenched as flow was reduced,Quenched,bass2-summary,104



DATASET: saffire_runs.csv
Shape: (20, 23)

Columns:
['id', 'flight', 'sample', 'material', 'material_verbatim', 'thickness_mm', 'width_cm', 'length_cm', 'flow_cm_s', 'flow_direction', 'pressure_kpa', 'o2_pct', 'o2_basis', 'burn_duration_s', 'spread_rate_mm_s', 'heat_release_avg_w', 'heat_release_peak_w', 'outcome', 'outcome_label', 'conditions_source', 'conditions_pdf_page', 'results_source', 'results_pdf_page']

First 3 rows:


,id,flight,sample,material,material_verbatim,thickness_mm,width_cm,length_cm,flow_cm_s,flow_direction,...,burn_duration_s,spread_rate_mm_s,heat_release_avg_w,heat_release_peak_w,outcome,outcome_label,conditions_source,conditions_pdf_page,results_source,results_pdf_page
0,saffire-1-1,Saffire-1,1-1,SIBAL fabric,Cotton-Fiberglass,0.37,40.6,94.0,20.0,concurrent,...,420.0,1.8,NaN,NaN,sustained,Burned until the airflow was shut off,saffire-1-3,29.0,saffire-1-3,29.0
1,saffire-1-2,Saffire-1,1-2,SIBAL fabric,Cotton-Fiberglass,0.37,40.6,10.0,20.0,opposed,...,70.0,1.3,NaN,NaN,sustained,Spread against the flow over the remaining fuel,saffire-1-3,29.0,saffire-1-3,29.0
2,saffire-2-1,Saffire-2,2-1,Silicone,Silicone,0.27,5.0,29.0,20.0,concurrent,...,NaN,NaN,NaN,NaN,extinguished,Did not spread; the small flame went out after...,saffire-1-3,29.0,saffire-1-3,29.0
